<a href="https://colab.research.google.com/github/Luisinho-31/data-analytics-pharma/blob/main/dashboard_distribuidor_farma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Dashboard de Ventas e Inventario — Distribuidora Farmacéutica

> **Problema que resuelve:** Muchas distribuidoras e importadoras manejan ventas, inventario y clientes en archivos de Excel dispersos. Eso genera reportes lentos, decisiones a ciegas y productos que se quedan sin stock sin aviso.
>
> Este análisis centraliza toda esa información, identifica tendencias, detecta alertas de inventario y predice la demanda futura por producto.

---
**Stack:** Python · Pandas · Plotly · Prophet  
**Autor:** Luis Fernando Castillo Garcia | [LinkedIn](https://www.linkedin.com/in/luis-fernando-castillo-garcia) | [GitHub](https://github.com/Luisinho-31)


##  0. Instalación de dependencias

In [ ]:
!pip install plotly prophet --quiet

##  1. Importar librerías

In [ ]:
# @title
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from prophet import Prophet
from datetime import datetime
import warnings
import ipywidgets as widgets
from IPython.display import display

warnings.filterwarnings('ignore')

print("✅ Librerías cargadas correctamente")

## 2. Generación de datos

> En un caso real, estos datos vendrían de un archivo Excel, una base de datos o un ERP.  
> Para este ejemplo generamos datos ficticios representativos de una distribuidora farmacéutica.


In [ ]:
# @title
import random
np.random.seed(42)
random.seed(42)

productos = [
    {"id": "P001", "nombre": "Paracetamol 500mg",   "categoria": "Analgésicos",       "precio": 45.0,  "stock_min": 100},
    {"id": "P002", "nombre": "Amoxicilina 500mg",    "categoria": "Antibióticos",      "precio": 120.0, "stock_min": 80},
    {"id": "P003", "nombre": "Ibuprofeno 400mg",     "categoria": "Analgésicos",       "precio": 55.0,  "stock_min": 100},
    {"id": "P004", "nombre": "Metformina 850mg",     "categoria": "Diabetes",          "precio": 90.0,  "stock_min": 60},
    {"id": "P005", "nombre": "Losartan 50mg",        "categoria": "Cardiovascular",    "precio": 75.0,  "stock_min": 60},
    {"id": "P006", "nombre": "Omeprazol 20mg",       "categoria": "Gastrointestinal",  "precio": 65.0,  "stock_min": 80},
    {"id": "P007", "nombre": "Atorvastatina 20mg",   "categoria": "Cardiovascular",    "precio": 110.0, "stock_min": 50},
    {"id": "P008", "nombre": "Azitromicina 500mg",   "categoria": "Antibióticos",      "precio": 140.0, "stock_min": 40},
    {"id": "P009", "nombre": "Vitamina C 1000mg",    "categoria": "Vitaminas",         "precio": 35.0,  "stock_min": 120},
    {"id": "P010", "nombre": "Insulina NPH",         "categoria": "Diabetes",          "precio": 280.0, "stock_min": 30},
]

clientes = [
    "Farmacia San Juan", "Clínica del Valle", "Hospital General Norte",
    "Farmacia Guadalajara", "Clínica Santa María", "Distribuidora MedPlus",
    "Farmacia Popular", "Centro Médico Sur", "Farmacias Similares CDMX",
    "Clínica Especialidades"
]
zonas = ["Norte", "Sur", "Este", "Oeste", "Centro"]

fechas = pd.date_range(end=datetime.today(), periods=365, freq='D')
ventas = []
for fecha in fechas:
    for _ in range(random.randint(5, 20)):
        p = random.choice(productos)
        cantidad = random.randint(10, 200)
        descuento = random.choice([0, 0, 0, 0.05, 0.10])
        ventas.append({
            "fecha": fecha,
            "producto_id": p["id"],
            "producto": p["nombre"],
            "categoria": p["categoria"],
            "cliente": random.choice(clientes),
            "zona": random.choice(zonas),
            "cantidad": cantidad,
            "precio_unitario": round(p["precio"] * (1 - descuento), 2),
            "total": round(cantidad * p["precio"] * (1 - descuento), 2),
            "descuento": descuento
        })

df_ventas = pd.DataFrame(ventas)

inventario = []
for p in productos:
    stock = random.randint(20, 500)
    inventario.append({
        "producto_id": p["id"],
        "producto": p["nombre"],
        "categoria": p["categoria"],
        "stock_actual": stock,
        "stock_minimo": p["stock_min"],
        "precio_unitario": p["precio"],
        "valor_inventario": round(stock * p["precio"], 2),
        "estado": "Crítico" if stock < p["stock_min"] else ("Bajo" if stock < p["stock_min"] * 1.5 else "Normal")
    })

df_inventario = pd.DataFrame(inventario)
print(f"✅ Datos generados: {len(df_ventas):,} registros de ventas | {len(df_inventario)} productos en inventario")
df_ventas.head(3)

## 3. KPIs Principales

> Métricas clave que un gerente necesita ver de un vistazo: ingresos, volumen, clientes activos y salud del inventario.


In [ ]:
# @title
ventas_total   = df_ventas["total"].sum()
unidades_total = df_ventas["cantidad"].sum()
clientes_act   = df_ventas["cliente"].nunique()
valor_inv      = df_inventario["valor_inventario"].sum()
prod_criticos  = len(df_inventario[df_inventario["estado"] == "Crítico"])
ticket_prom    = df_ventas.groupby("cliente")["total"].sum().mean()

kpis = pd.DataFrame({
    "KPI": ["💰 Ventas Totales (MXN)", "📦 Unidades Vendidas", "🏪 Clientes Activos",
            "🏦 Valor Inventario (MXN)", "⚠️ Productos Críticos", "🧾 Ticket Promedio (MXN)"],
    "Valor": [f"${ventas_total:,.0f}", f"{unidades_total:,}", f"{clientes_act}",
              f"${valor_inv:,.0f}", f"{prod_criticos}", f"${ticket_prom:,.0f}"]
})

fig_kpi = go.Figure(data=[go.Table(
    columnwidth=[250, 150],
    header=dict(
        values=["<b>Indicador</b>", "<b>Valor</b>"],
        fill_color="#1e2130", font=dict(color="#4ecca3", size=13),
        align="left", height=35
    ),
    cells=dict(
        values=[kpis["KPI"], kpis["Valor"]],
        fill_color=[["#252837", "#1e2130"] * 10],
        font=dict(color=["#ccd6f6", "#f7b731"], size=13),
        align="left", height=32
    )
)])
fig_kpi.update_layout(
    title="Resumen Ejecutivo — Últimos 12 meses",
    title_font=dict(color="#ccd6f6", size=15),
    paper_bgcolor="#0f1117", margin=dict(t=50, b=10, l=10, r=10), height=280
)
fig_kpi.show()

##  4. Tendencia de Ventas

> Visualizar la tendencia diaria con una media móvil de 7 días permite identificar picos, caídas estacionales y el ritmo general del negocio — algo que en Excel requiere fórmulas manuales, tablas dinámicas y tiempo considerable cada vez que se actualiza la información.


In [ ]:
# @title
ventas_dia = df_ventas.groupby("fecha")["total"].sum().reset_index()
ventas_dia["media_movil_7d"] = ventas_dia["total"].rolling(7, min_periods=1).mean()

fig_tend = go.Figure()
fig_tend.add_trace(go.Scatter(
    x=ventas_dia["fecha"], y=ventas_dia["total"],
    mode="lines", name="Ventas diarias",
    line=dict(color="#4ecca3", width=1.5),
    fill="tozeroy", fillcolor="rgba(78,204,163,0.08)"
))
fig_tend.add_trace(go.Scatter(
    x=ventas_dia["fecha"], y=ventas_dia["media_movil_7d"],
    mode="lines", name="Media móvil 7 días",
    line=dict(color="#f7b731", width=2.5, dash="dot")
))
fig_tend.update_layout(
    title="Ventas Diarias con Media Móvil de 7 Días",
    title_font=dict(color="#ccd6f6", size=15),
    plot_bgcolor="#1e2130", paper_bgcolor="#0f1117",
    font=dict(color="#8892b0"),
    xaxis=dict(gridcolor="#2d3250", title="Fecha"),
    yaxis=dict(gridcolor="#2d3250", title="Ventas (MXN)", tickprefix="$"),
    legend=dict(bgcolor="#1e2130"),
    height=400
)
fig_tend.show()

##  5. Análisis por Categoría y Zona

> Identificar qué categorías de productos y qué zonas geográficas generan más ingresos permite enfocar esfuerzos de venta y distribución donde más impactan.


In [ ]:
# @title
fig_sub = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ventas por Categoría", "Ventas por Zona"),
    specs=[[{"type": "bar"}, {"type": "pie"}]]
)

cat_ventas = df_ventas.groupby("categoria")["total"].sum().sort_values().reset_index()
fig_sub.add_trace(go.Bar(
    x=cat_ventas["total"], y=cat_ventas["categoria"],
    orientation="h", marker_color="#4ecca3",
    text=cat_ventas["total"].apply(lambda x: f"${x:,.0f}"),
    textposition="outside"
), row=1, col=1)

zona_ventas = df_ventas.groupby("zona")["total"].sum().reset_index()
fig_sub.add_trace(go.Pie(
    values=zona_ventas["total"], labels=zona_ventas["zona"],
    marker_colors=["#4ecca3", "#f7b731", "#ff6b6b", "#a29bfe", "#74b9ff"],
    hole=0.45
), row=1, col=2)

fig_sub.update_layout(
    title="Distribución de Ventas por Categoría y Zona Geográfica",
    title_font=dict(color="#ccd6f6", size=15),
    plot_bgcolor="#1e2130", paper_bgcolor="#0f1117",
    font=dict(color="#8892b0"),
    showlegend=True,
    legend=dict(bgcolor="#1e2130"),
    height=420
)
fig_sub.update_xaxes(gridcolor="#2d3250", tickprefix="$", row=1, col=1)
fig_sub.show()

##  6. Top 10 Productos

> Conocer qué productos generan más ingresos (y cuántas unidades representan) ayuda a priorizar el reabastecimiento y las negociaciones con proveedores.


In [ ]:
# @title
top_prod = df_ventas.groupby("producto").agg(
    total=("total","sum"),
    unidades=("cantidad","sum")
).sort_values("total", ascending=False).head(10).reset_index()

fig_top = px.bar(
    top_prod, x="total", y="producto", orientation="h",
    color="unidades",
    color_continuous_scale=["#2d3250", "#4ecca3"],
    labels={"total": "Ventas (MXN)", "producto": "", "unidades": "Unidades vendidas"},
    text=top_prod["total"].apply(lambda x: f"${x:,.0f}")
)
fig_top.update_traces(textposition="outside")
fig_top.update_layout(
    title="Top 10 Productos por Ingresos",
    title_font=dict(color="#ccd6f6", size=15),
    plot_bgcolor="#1e2130", paper_bgcolor="#0f1117",
    font=dict(color="#8892b0"),
    xaxis=dict(gridcolor="#2d3250", tickprefix="$"),
    yaxis=dict(gridcolor="rgba(0,0,0,0)"),
    coloraxis_colorbar=dict(title="Unidades", tickfont=dict(color="#8892b0")),
    height=420
)
fig_top.show()

##  7. Estado del Inventario y Alertas

> Un quiebre de stock puede significar ventas perdidas y clientes insatisfechos. Este módulo identifica automáticamente productos en estado **Crítico** (por debajo del mínimo) o **Bajo** (menos del 150% del mínimo) para que el equipo actúe antes de que sea tarde.


In [ ]:
# @title
# Tabla de alertas
criticos = df_inventario[df_inventario["estado"].isin(["Crítico", "Bajo"])].sort_values("stock_actual")
print("⚠️  ALERTAS DE INVENTARIO")
print("=" * 60)
for _, row in criticos.iterrows():
    emoji = "🔴" if row["estado"] == "Crítico" else "🟡"
    print(f"{emoji} {row['producto']:<30} Stock: {row['stock_actual']:>4} | Mínimo: {row['stock_minimo']:>4} | Estado: {row['estado']}")

# Gráfica
colores_estado = {"Normal": "#4ecca3", "Bajo": "#ffd166", "Crítico": "#ff6b6b"}
fig_inv = go.Figure()

for estado, color in colores_estado.items():
    df_est = df_inventario[df_inventario["estado"] == estado]
    if len(df_est):
        fig_inv.add_trace(go.Bar(
            name=estado, x=df_est["producto"], y=df_est["stock_actual"],
            marker_color=color,
            text=df_est["stock_actual"], textposition="outside"
        ))

fig_inv.add_trace(go.Scatter(
    x=df_inventario["producto"], y=df_inventario["stock_minimo"],
    mode="lines+markers", name="Stock Mínimo",
    line=dict(color="#ff6b6b", dash="dash", width=2),
    marker=dict(size=8, symbol="diamond")
))

fig_inv.update_layout(
    barmode="group",
    title="Stock Actual vs Stock Mínimo por Producto",
    title_font=dict(color="#ccd6f6", size=15),
    plot_bgcolor="#1e2130", paper_bgcolor="#0f1117",
    font=dict(color="#8892b0"),
    xaxis=dict(gridcolor="#2d3250", tickangle=-25),
    yaxis=dict(gridcolor="#2d3250", title="Unidades en Stock"),
    legend=dict(bgcolor="#1e2130"),
    height=420
)
fig_inv.show()

##  8. Predicción de Demanda con Prophet

> Predecir cuántas unidades se venderán en los próximos días permite planear reabastecimientos con anticipación, evitar quiebres de stock y negociar mejor con proveedores.
>
> Usamos **Prophet** (Meta), un modelo de series de tiempo robusto que captura estacionalidad semanal y anual automáticamente.


In [ ]:
# @title

producto_widget = widgets.Dropdown(
    options=sorted(df_ventas["producto"].unique().tolist()),
    value="Paracetamol 500mg",
    description="Producto:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px")
)

dias_widget = widgets.IntSlider(
    value=30, min=7, max=90, step=1,
    description="Días a predecir:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

display(producto_widget, dias_widget)

In [ ]:
# @title
PRODUCTO_ANALIZAR = producto_widget.value
DIAS_PREDICCION = dias_widget.value

df_p = (
    df_ventas[df_ventas["producto"] == PRODUCTO_ANALIZAR]
    .groupby("fecha")["cantidad"].sum()
    .reset_index()
    .rename(columns={"fecha": "ds", "cantidad": "y"})
)

modelo = Prophet(
    changepoint_prior_scale=0.05,
    seasonality_prior_scale=10,
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False
)
modelo.fit(df_p)
futuro   = modelo.make_future_dataframe(periods=DIAS_PREDICCION)
forecast = modelo.predict(futuro)

ultimo_hist    = df_p["ds"].max()
pred_futuro    = forecast[forecast["ds"] > ultimo_hist]
demanda_total  = int(pred_futuro["yhat"].sum())
demanda_diaria = demanda_total / DIAS_PREDICCION

stock_prod = df_inventario[df_inventario["producto"] == PRODUCTO_ANALIZAR]["stock_actual"].values[0]
dias_cob   = int(stock_prod / demanda_diaria) if demanda_diaria > 0 else 0

print(f"📦 Producto analizado : {PRODUCTO_ANALIZAR}")
print(f"🔮 Demanda estimada   : {demanda_total:,} unidades en los próximos {DIAS_PREDICCION} días")
print(f"📅 Días de cobertura  : {dias_cob} días con stock actual de {stock_prod} unidades")
print(f"{'⚠️  REORDEN RECOMENDADO PRONTO' if dias_cob < 15 else '✅ Stock suficiente por ahora'}")

In [ ]:
# @title
fig_pred = go.Figure()
fig_pred.add_trace(go.Scatter(
    x=df_p["ds"], y=df_p["y"],
    mode="lines", name="Ventas reales",
    line=dict(color="#4ecca3", width=2)
))
fig_pred.add_trace(go.Scatter(
    x=forecast["ds"], y=forecast["yhat"],
    mode="lines", name="Predicción",
    line=dict(color="#f7b731", width=2.5, dash="dot")
))

ds_concat = pd.concat([forecast["ds"].reset_index(drop=True),
                        forecast["ds"].iloc[::-1].reset_index(drop=True)])
y_concat  = pd.concat([forecast["yhat_upper"].reset_index(drop=True),
                        forecast["yhat_lower"].iloc[::-1].reset_index(drop=True)])

fig_pred.add_trace(go.Scatter(
    x=ds_concat, y=y_concat,
    fill="toself", fillcolor="rgba(247,183,49,0.1)",
    line=dict(color="rgba(0,0,0,0)"),
    name="Intervalo de confianza"
))

fig_pred.add_shape(
    type="line",
    x0=str(ultimo_hist), x1=str(ultimo_hist),
    y0=0, y1=1,
    yref="paper",
    line=dict(color="#a29bfe", dash="dash", width=1.5)
)
fig_pred.add_annotation(
    x=str(ultimo_hist), y=1,
    yref="paper",
    text="Hoy →",
    showarrow=False,
    font=dict(color="#a29bfe"),
    xanchor="left"
)

fig_pred.update_layout(
    title=f"Predicción de Demanda — {PRODUCTO_ANALIZAR} ({DIAS_PREDICCION} días)",
    title_font=dict(color="#ccd6f6", size=15),
    plot_bgcolor="#1e2130", paper_bgcolor="#0f1117",
    font=dict(color="#8892b0"),
    xaxis=dict(gridcolor="#2d3250", title="Fecha"),
    yaxis=dict(gridcolor="#2d3250", title="Unidades"),
    legend=dict(bgcolor="#1e2130"),
    height=420
)
fig_pred.show()

##  9. Análisis de Clientes

> Identificar qué clientes generan más ingresos permite priorizar la atención, negociar descuentos estratégicos y detectar clientes en riesgo de abandono.


In [ ]:
# @title
top_clientes = df_ventas.groupby("cliente").agg(
    total=("total","sum"),
    pedidos=("cantidad","count"),
    unidades=("cantidad","sum")
).sort_values("total", ascending=False).reset_index()

top_clientes["% del total"] = (top_clientes["total"] / top_clientes["total"].sum() * 100).round(1)

fig_cli = go.Figure(data=[go.Table(
    columnwidth=[200, 120, 100, 100, 100],
    header=dict(
        values=["<b>Cliente</b>","<b>Ventas (MXN)</b>","<b>Pedidos</b>","<b>Unidades</b>","<b>% del total</b>"],
        fill_color="#1e2130", font=dict(color="#4ecca3", size=12),
        align="left", height=35
    ),
    cells=dict(
        values=[
            top_clientes["cliente"],
            top_clientes["total"].apply(lambda x: f"${x:,.0f}"),
            top_clientes["pedidos"].apply(lambda x: f"{x:,}"),
            top_clientes["unidades"].apply(lambda x: f"{x:,}"),
            top_clientes["% del total"].apply(lambda x: f"{x}%")
        ],
        fill_color=[["#252837","#1e2130"]*10],
        font=dict(color="#ccd6f6", size=12),
        align="left", height=30
    )
)])
fig_cli.update_layout(
    title="Ranking de Clientes por Ingresos",
    title_font=dict(color="#ccd6f6", size=15),
    paper_bgcolor="#0f1117",
    margin=dict(t=50, b=10), height=380
)
fig_cli.show()

---

##  10. Conclusiones y Próximos Pasos

Este dashboard demuestra cómo transformar datos dispersos en decisiones claras:

| Módulo | Insight clave |
|---|---|
| **KPIs** | Visibilidad inmediata de salud del negocio |
| **Tendencia** | Detección de estacionalidad y caídas |
| **Categorías/Zonas** | Foco en lo que más vende y dónde |
| **Inventario** | Prevención de quiebres de stock |
| **Predicción** | Planificación proactiva de reabastecimiento |
| **Clientes** | Priorización de cuentas clave |

### 🔧 Adaptaciones para un caso real
- Conectar directamente a archivos Excel del cliente
- Integrar con base de datos (MySQL, PostgreSQL, BigQuery)
- Automatizar reportes diarios por correo
- Agregar módulo de rentabilidad por producto/cliente

---
**¿Tu empresa necesita algo similar?**  
📧 castillo.luisf94@gmail.com  
💼 [LinkedIn](https://www.linkedin.com/in/luis-fernando-castillo-garcia)
